# Projekt 2: Sprach-Werkstatt

**Teammitglieder:**  
- Gabriel Deiac
- Laurenz Berchtold

**Projektwahl:** Sprach-Werkstatt (Schreibassistenz mit HuggingFace + Gradio)

---
Dieses Notebook enthält eine lauffähige Gradio-App mit:
- Umformulieren (text-generation)
- Wort-Vorschläge mit [MASK] (fill-mask)
- Übersetzen DE↔EN, DE↔ES und DE↔IT (jeweils in beide Richtungen)
- Ton-Check (sentiment-analysis)

Zusätzlich: eigenes CSS-Design, Error Handling und History via gr.State.

## Kurzdokumentation

Die Sprach-Werkstatt unterstützt beim Schreiben durch mehrere NLP-Funktionen in einer Oberfläche.  
Für das Umformulieren verwenden wir eine text-generation Pipeline mit Stil-Auswahl (formell, informell, einfach, wissenschaftlich).  
Für Wort-Vorschläge nutzen wir fill-mask, wobei ein Satz mit [MASK] eingegeben wird und passende Wörter samt Wahrscheinlichkeit angezeigt werden.  
Die Übersetzung erfolgt mit translation zwischen Deutsch und Englisch, Spanisch sowie Italienisch in beide Richtungen.  
Zusätzlich bewertet ein Ton-Check per sentiment-analysis, ob der Text eher positiv, neutral oder negativ wirkt.  
Die App wurde mit gr.Blocks und mehreren Tabs umgesetzt.  
Ein eigenes CSS sorgt für ein konsistentes Editor/Schreibtisch-Design mit Hover-Effekten und gestylten Buttons.  
Fehleingaben wie leere oder zu kurze Texte werden abgefangen und als klare Hinweise ausgegeben.  
Als Bonus speichern wir die Umformulierungen in einer Session-History mit gr.State.

In [1]:
import os
import re
import pandas as pd
import gradio as gr
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

load_dotenv()
hf_token = os.getenv("HF_TOKEN")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("HF token gefunden: authentifizierter Download aktiv.")
else:
    print("Kein HF_TOKEN in .env oder Umgebung gesetzt: Download ohne Token.")

print("Lade Modelle...")


def load_text_generation_pipeline():
    candidates = [
        "Qwen/Qwen2.5-0.5B-Instruct",
        "distilgpt2",
    ]
    last_error = None
    for model_name in candidates:
        try:
            generator = pipeline("text-generation", model=model_name, token=hf_token)
            print(f"Text-Generation-Modell geladen: {model_name}")
            return generator
        except Exception as exc:
            last_error = exc
            print(f"Text-Generation-Modell {model_name} konnte nicht geladen werden: {exc}")
    raise RuntimeError("Kein Text-Generation-Modell konnte geladen werden.") from last_error


text_gen = load_text_generation_pipeline()
fill_mask = pipeline("fill-mask", model="bert-base-german-cased", token=hf_token)
sentiment_pipe = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", token=hf_token)


def load_translation_model(model_name: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, token=hf_token)
    return tokenizer, model


de_en_tok, de_en_model = load_translation_model("Helsinki-NLP/opus-mt-de-en")
en_de_tok, en_de_model = load_translation_model("Helsinki-NLP/opus-mt-en-de")
de_es_tok, de_es_model = load_translation_model("Helsinki-NLP/opus-mt-de-es")
es_de_tok, es_de_model = load_translation_model("Helsinki-NLP/opus-mt-es-de")
de_it_tok, de_it_model = load_translation_model("Helsinki-NLP/opus-mt-de-it")
it_de_tok, it_de_model = load_translation_model("Helsinki-NLP/opus-mt-it-de")

print("Modelle geladen.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF token gefunden: authentifizierter Download aktiv.
Lade Modelle...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Text-Generation-Modell geladen: Qwen/Qwen2.5-0.5B-Instruct


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Modelle geladen.


In [2]:
STYLE_HINTS = {
    "formell": "formal und prägnant",
    "informell": "locker und natürlich",
    "einfach": "in sehr einfacher Sprache",
    "wissenschaftlich": "sachlich und wissenschaftlich",
}

SUMMARY_HINTS = {
    "DE -> EN": "Summarize the following English text in one short sentence. Output only the summary.",
    "EN -> DE": "Fasse den folgenden deutschen Text in einem kurzen Satz zusammen. Gib nur die Zusammenfassung aus.",
    "DE -> ES": "Resume el siguiente texto en una sola frase corta. Devuelve solo el resumen.",
    "ES -> DE": "Fasse den folgenden spanischen Text in einem kurzen Satz zusammen. Gib nur die Zusammenfassung aus.",
    "DE -> IT": "Riassumi il seguente testo in una sola frase breve. Restituisci solo il riassunto.",
    "IT -> DE": "Fasse den folgenden italienischen Text in einem kurzen Satz zusammen. Gib nur die Zusammenfassung aus.",
}


def _clean_text(x: str) -> str:
    return (x or "").strip()


def _strip_prompt_echo(output: str, prompt: str | None = None) -> str:
    text = _clean_text(output)
    if prompt:
        prompt = prompt.strip()
        if text.startswith(prompt):
            text = text[len(prompt):].strip()
    for prefix in ("Umformulierung:", "Zusammenfassung:", "Antwort:", "Result:", "Rewritten text:", "Summary:"):
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    return text.strip(' \n"\'')


def _generate_text(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = text_gen.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: value.to(text_gen.model.device) for key, value in inputs.items()}
    generated_ids = text_gen.model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=getattr(text_gen.tokenizer, "eos_token_id", 50256),
    )
    input_length = inputs["input_ids"].shape[-1]
    generated = text_gen.tokenizer.decode(generated_ids[0][input_length:], skip_special_tokens=True)
    return _strip_prompt_echo(generated, prompt)


def _short_summary(text: str, max_words: int = 20) -> str:
    text = _clean_text(text)
    if not text:
        return "—"

    sentences = re.split(r"(?<=[.!?])\s+", text)
    candidate = sentences[0]
    if len(candidate.split()) > max_words and len(sentences) > 1:
        candidate = " ".join(sentences[:2])

    words = candidate.split()
    if len(words) <= max_words:
        return candidate.strip()
    return " ".join(words[:max_words]).rstrip(" ,;:") + " ..."


def _history_to_md(history):
    if not history:
        return "_Noch keine Umformulierungen in dieser Session._"
    lines = ["### History (Session)"]
    for i, item in enumerate(history[-10:], start=1):
        lines.append(f"{i}. **{item['style']}**  \n   ➜ {item['result']}")
    return "\n".join(lines)


def reformulate_text(text, style, history):
    text = _clean_text(text)
    history = history or []

    if not text:
        return "Bitte Text eingeben.", history, _history_to_md(history)
    if len(text) < 8:
        return "Text ist zu kurz (mindestens 8 Zeichen).", history, _history_to_md(history)

    style_hint = STYLE_HINTS.get(style, STYLE_HINTS["formell"])
    prompt = (
        "Du bist ein präziser deutscher Schreibassistent.\n"
        f"Formuliere den folgenden Text {style_hint} um.\n"
        "Ändere Wortwahl und Satzbau, aber behalte die Bedeutung bei.\n"
        "Gib nur den umformulierten Text aus.\n\n"
        f"Text:\n{text}\n\nUmformulierung:"
    )
    result = _generate_text(prompt, max_new_tokens=96)

    if not result:
        result = text

    history.append({"style": style, "result": result})
    return result, history, _history_to_md(history)


def clear_history():
    history = []
    return history, _history_to_md(history)


def predict_mask(sentence):
    sentence = _clean_text(sentence)
    if not sentence:
        return pd.DataFrame([{"Wort": "—", "Wahrscheinlichkeit": "Bitte Satz eingeben."}])
    if "[MASK]" not in sentence:
        return pd.DataFrame([{"Wort": "—", "Wahrscheinlichkeit": "Bitte [MASK] im Satz verwenden."}])

    preds = fill_mask(sentence, top_k=8)
    rows = []
    for p in preds:
        token = p["token_str"].strip()
        prob = f"{p['score'] * 100:.2f}%"
        example = p.get("sequence", "").strip()
        rows.append({"Wort": token, "Wahrscheinlichkeit": prob, "Beispielsatz": example})
    return pd.DataFrame(rows)


def _run_translation(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    generated = model.generate(**inputs, max_length=256)
    return tokenizer.decode(generated[0], skip_special_tokens=True)


def _summarize_translation(translated: str, direction: str) -> str:
    translated = _clean_text(translated)
    if not translated:
        return "—"

    prompt = SUMMARY_HINTS.get(
        direction,
        "Fasse den folgenden Text in einem kurzen Satz zusammen. Gib nur die Zusammenfassung aus.",
    )
    summary_prompt = f"{prompt}\n\nText:\n{translated}\n\nZusammenfassung:"
    summary = _generate_text(summary_prompt, max_new_tokens=48)

    if not summary:
        summary = _short_summary(translated, max_words=18)
    return summary


def translate_text(text, direction):
    text = _clean_text(text)
    if not text:
        return "Bitte Text eingeben.", "—"

    translators = {
        "DE -> EN": (de_en_tok, de_en_model),
        "EN -> DE": (en_de_tok, en_de_model),
        "DE -> ES": (de_es_tok, de_es_model),
        "ES -> DE": (es_de_tok, es_de_model),
        "DE -> IT": (de_it_tok, de_it_model),
        "IT -> DE": (it_de_tok, it_de_model),
    }

    selected = translators.get(direction)
    if selected is None:
        return "Ungültige Richtung gewählt.", "—"

    tokenizer, model = selected
    translated = _run_translation(text, tokenizer, model)
    summary = _summarize_translation(translated, direction)
    return translated, summary


def tone_check(text):
    text = _clean_text(text)
    if not text:
        return "Bitte Text eingeben."
    if len(text) < 4:
        return "Text ist zu kurz für Ton-Analyse."

    pred = sentiment_pipe(text[:512])[0]
    label = pred["label"]
    score = pred["score"]

    mapping = {
        "positive": "Positiv",
        "negative": "Negativ",
        "neutral": "Neutral",
        "LABEL_0": "Negativ",
        "LABEL_1": "Neutral",
        "LABEL_2": "Positiv",
    }

    nice_label = mapping.get(label.lower(), mapping.get(label, label))
    return f"**Ton-Check:** {nice_label} ({score * 100:.1f}%)"

In [3]:
custom_css = """
.gradio-container {
  background: var(--background-fill-primary);
  color: var(--body-text-color);
}

#title-box {
  background: var(--block-background-fill);
  border: 1px solid var(--border-color-primary);
  border-radius: 14px;
  padding: 14px;
  box-shadow: var(--shadow-drop-lg);
}

button.primary {
  background: var(--color-accent) !important;
  border: none !important;
  color: var(--color-accent-inverted, white) !important;
  transition: transform .12s ease, background .2s ease, box-shadow .2s ease;
}

button.primary:hover {
  background: var(--color-accent-soft) !important;
  transform: translateY(-1px);
  box-shadow: var(--shadow-drop-lg);
}

button.secondary {
  border-color: var(--border-color-primary) !important;
  color: var(--body-text-color) !important;
}

#typewriter textarea {
  font-family: "Courier New", monospace !important;
}

#tool-card {
  background: var(--block-background-fill);
  border: 1px solid var(--border-color-primary);
  border-radius: 12px;
  padding: 12px;
  box-shadow: var(--shadow-drop-sm);
}
"""


with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Sprach-Werkstatt") as demo:
    history_state = gr.State([])

    gr.Markdown("## Sprach-Werkstatt", elem_id="title-box")
    gr.Markdown("Schreiben, umformulieren, uebersetzen und Ton pruefen - alles in einer App.")

    with gr.Row():
        with gr.Column(scale=4):
            with gr.Tabs():
                with gr.Tab("Umformulieren"):
                    with gr.Row():
                        with gr.Column(scale=2):
                            in_text = gr.Textbox(label="Text", placeholder="Gib hier deinen Text ein ...", lines=6)
                            style = gr.Dropdown(
                                choices=["formell", "informell", "einfach", "wissenschaftlich"],
                                value="formell",
                                label="Stil"
                            )
                            btn_rewrite = gr.Button("Umformulieren", variant="primary")
                            out_text = gr.Textbox(label="Ergebnis", lines=8, elem_id="typewriter")

                            btn_tone = gr.Button("Ton-Check", variant="secondary")
                            out_tone = gr.Markdown()

                        with gr.Column(scale=1):
                            hist_md = gr.Markdown("_Noch keine Umformulierungen in dieser Session._")
                            btn_clear = gr.Button("History leeren")

                    btn_rewrite.click(
                        reformulate_text,
                        inputs=[in_text, style, history_state],
                        outputs=[out_text, history_state, hist_md],
                    )
                    btn_clear.click(clear_history, outputs=[history_state, hist_md])
                    btn_tone.click(tone_check, inputs=in_text, outputs=out_tone)

                with gr.Tab("Wort-Vorschlaege ([MASK])"):
                    mask_in = gr.Textbox(
                        label="Satz mit [MASK]",
                        placeholder="Beispiel: Das Wetter ist heute [MASK].",
                        lines=3
                    )
                    mask_btn = gr.Button("Vorschlaege anzeigen", variant="primary")
                    mask_out = gr.Dataframe(label="Top-Vorschlaege")
                    mask_btn.click(predict_mask, inputs=mask_in, outputs=mask_out)
                    gr.Examples(
                        examples=[
                            ["Das Wetter ist heute [MASK]."],
                            ["Die Loesung ist wirklich [MASK]."],
                            ["Das Ergebnis war [MASK] und ueberzeugend."],
                        ],
                        inputs=mask_in,
                    )

                with gr.Tab("Uebersetzer"):
                    tr_in = gr.Textbox(label="Text", placeholder="Text fuer die Uebersetzung ...", lines=5)
                    direction = gr.Radio(
                        choices=["DE -> EN", "EN -> DE", "DE -> ES", "ES -> DE", "DE -> IT", "IT -> DE"],
                        value="DE -> EN",
                        label="Richtung"
                    )
                    tr_btn = gr.Button("Uebersetzen", variant="primary")
                    tr_out = gr.Textbox(label="Uebersetzung", lines=6, elem_id="typewriter")
                    tr_sum = gr.Textbox(label="Kurzzusammenfassung", lines=2)
                    tr_btn.click(translate_text, inputs=[tr_in, direction], outputs=[tr_out, tr_sum])

        with gr.Column(scale=1, min_width=220):
            gr.Markdown("### Werkzeugleiste")
            gr.Markdown("- Umformulieren\n- Wort-Vorschlaege\n- Uebersetzen\n- Ton-Check")
            gr.Markdown(
                "Nutze die Gradio-Theme-Einstellung oben rechts fuer Light/Dark, statt hier einen eigenen Schalter zu haben."
            )
            gr.Markdown("Tipp: Starte mit einem klaren Satz und pruefe danach den Ton.")

demo.launch()

/tmp/ipykernel_28453/2256876364.py:47: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, theme=gr.themes.Soft(), title="Sprach-Werkstatt") as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
